# Mapa de ataques hacker reales

**Proyecto final — DataXperience, Universidad EAN**
**Autor:** Sebastian Renteria Cruz

Un *honeypot* es un servidor expuesto a propósito en internet para registrar cada intento de ataque.
Este cuaderno analiza unos 450,000 ataques registrados por honeypots de AWS entre marzo y septiembre de 2013.

**Pregunta del proyecto:** ¿desde dónde, cuándo y contra qué servicios atacan los hackers,
y se puede predecir qué servicio van a atacar?

**Relación con la ciberseguridad:** el análisis de amenazas es la base del trabajo de un analista SOC.
Sirve para decidir qué puertos cerrar, qué servicios vigilar y a qué horas reforzar el monitoreo.

## Índice

1. Importaciones
2. Carga de datos
3. **Etapa 1 — Fundamentos y preparación de datos**
    1. Primera exploración
    2. Tipos de variables
    3. Detección de problemas *(siguiente paso)*
    4. Limpieza *(siguiente paso)*
4. Etapa 2 — Análisis estadístico *(pendiente)*
5. Etapa 3 — Modelo predictivo *(pendiente)*
6. Conclusiones *(pendiente)*

## 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Mostrar todas las columnas al imprimir tablas anchas
pd.set_option("display.max_columns", None)

## 2. Carga de datos

El CSV se busca primero en la carpeta `datos/` del repositorio (ejecución local).
Si no está, como pasa en un Colab nuevo, se descarga desde la URL *raw* de GitHub.

In [ ]:
NOMBRE_CSV = "AWS_Honeypot_marx-geo.csv"

URL_RAW = f"https://raw.githubusercontent.com/RenteriaSc/mapa-ataques-honeypot/main/datos/{NOMBRE_CSV}"

# El cuaderno vive en notebook/, así que los datos están un nivel arriba
ruta_local = Path("..") / "datos" / NOMBRE_CSV
origen = ruta_local if ruta_local.exists() else URL_RAW

df_original = pd.read_csv(origen, low_memory=False)
print(f"Datos cargados desde: {origen}")

## 3. Etapa 1 — Fundamentos y preparación de datos

### 3.1 Primera exploración

Antes de cambiar nada, se mira el tamaño, los tipos de cada columna, unas filas de ejemplo
y las estadísticas básicas.

In [ ]:
filas, columnas = df_original.shape
print(f"Filas: {filas:,}  |  Columnas: {columnas}")

In [ ]:
df_original.info()

In [ ]:
df_original.head()

In [ ]:
df_original.describe()

**Interpretación:** *(se completa después de ejecutar las celdas)*

### 3.2 Tipos de variables

Clasificar cada columna dice qué gráficas y qué tratamiento le corresponden:

| Tipo | Columnas | Ejemplo |
|---|---|---|
| Fecha | `datetime` | 3/3/13 21:53 |
| Categórica | `host`, `proto`, `cc`, `country`, `locale`, `localeabbr`, `postalcode` | groucho-oregon, TCP, CN |
| Numérica (identificador) | `spt`, `dpt` | puertos 6000, 1433 |
| Numérica (coordenada) | `latitude`, `longitude` | 28.55, 115.93 |
| Identificador de IP | `src` (IP como número), `srcstr` (IP como texto) | 61.131.218.218 |
| Por revisar | `type` y la columna extra sin nombre | — |

Los puertos son números, pero no se pueden promediar: el puerto 1433 no es "más" que el 22.
Son etiquetas de servicio, así que en el análisis se tratan como categorías.

In [ ]:
# Cuántos valores distintos tiene cada columna: ayuda a confirmar la tabla anterior
df_original.nunique().sort_values()

**Interpretación:** *(se completa después de ejecutar las celdas)*